In [1]:
import pandas as pd
import os
import numpy as np
from datetime import datetime
import ast
from googletrans import Translator
import asyncio, random
from openai import OpenAI
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor, as_completed
from itertools import chain
import numpy as np
import pandas as pd
from transformers import pipeline
from concurrent.futures import ThreadPoolExecutor, as_completed
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
import torch

# Load

"Pablo Picasso"
"Pierre-Auguste Renoir"
"Käthe Kollwitz"
"Max Ernst"
"Karel Appel"

In [39]:
threshold = 0.70

In [40]:
Artist_name ="Pablo Picasso"
df1 = pd.read_excel(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Codes\\Evaluation\\golden_set_move_creative_{int(threshold*100)}_{Artist_name.split(" ")[-1]}.xlsx")
Artist_name ="Pierre-Auguste Renoir"
df2 = pd.read_excel(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Codes\\Evaluation\\golden_set_move_creative_{int(threshold*100)}_{Artist_name.split(" ")[-1]}.xlsx")
Artist_name ="Käthe Kollwitz"
df3 = pd.read_excel(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Codes\\Evaluation\\golden_set_move_creative_{int(threshold*100)}_{Artist_name.split(" ")[-1]}.xlsx")
Artist_name ="Max Ernst"
df4 = pd.read_excel(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Codes\\Evaluation\\golden_set_move_creative_{int(threshold*100)}_{Artist_name.split(" ")[-1]}.xlsx")
Artist_name ="Karel Appel"
df5 = pd.read_excel(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Codes\\Evaluation\\golden_set_move_creative_{int(threshold*100)}_{Artist_name.split(" ")[-1]}.xlsx")

In [41]:
print(f"shape for Picasso: {df1.shape}")
print(f"shape for Renoir: {df2.shape}")
print(f"shape for Kollwitz: {df3.shape}")
print(f"shape for Ernst: {df4.shape}")
print(f"shape for Appel: {df5.shape}")

shape for Picasso: (1084, 24)
shape for Renoir: (189, 24)
shape for Kollwitz: (589, 24)
shape for Ernst: (6, 24)
shape for Appel: (6, 24)


In [42]:
gold_set = pd.concat([df1, df2,df3,df4,df5], axis=0, ignore_index=True)

In [43]:
gold_set.to_excel(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Codes\\Evaluation\\golden_set_move_creative_{int(threshold*100)}.xlsx")

In [44]:
gold_set.shape

(1874, 24)

# Comment Score

In [5]:
classifier = pipeline("text-classification", model="yangheng/deberta-v3-base-absa-v1.1")

C:\Users\MissTiny\anaconda3\envs\Creativity\Lib\site-packages\transformers\convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Device set to use cuda:0


In [61]:
def compute_creativity(i,row,classifier):
    score_claude= classifier(row.creativity_comment_claude, text_pair="creativ")
    score_gemini=classifier(row.creativity_comment_gemini, text_pair="creativ")
    score_openai=classifier(row.creativity_comment, text_pair="creativ")
    row_score = []
    if score_claude[0]['label']=='Positive':
        row_score.append(score_claude[0]['score'])
    else:
        row_score.append(0)
    if score_gemini[0]['label']=='Positive':
        row_score.append(score_gemini[0]['score'])
    else:
        row_score.append(0)
    if score_openai[0]['label']=='Positive':
        row_score.append(score_openai[0]['score'])
    else:
        row_score.append(0)
    def softmax(logits):
    logits = np.array(logits, dtype=float)
    shifted = logits - np.max(logits)        # stability trick
    exp_vals = np.exp(shifted)
    return exp_vals / np.sum(exp_vals)
    return i, np.mean(row_score)

In [62]:
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,gold_set.shape[0])
N = min(number_size, gold_set.shape[0]-range_start)
max_workers = 20
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
while range_start < gold_set.shape[0]:
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Now at {range_start}")
    creativity = np.zeros(N)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(compute_creativity, i, 
                    gold_set.iloc[i],
                    classifier): i
            for i in range(range_start,range_end)
        }
        
        completed = 0
        for fut in as_completed(futures):
            i,row_creativity = fut.result()
            creativity[i-range_start]  = row_creativity
            completed += 1
            if completed % 100 == 0:
                print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Completed {completed}/{gold_set.shape[0]}")
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Saving embeddings")
    np.save(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\creativity_70.npy", creativity)
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Embedding Saved")
    range_start = range_start + N
    range_end = min(range_start+number_size,gold_set.shape[0])
    N = min(number_size, gold_set.shape[0]-range_start)
    
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

2025-12-09 07:13:50: Start
2025-12-09 07:13:50: Now at 0
2025-12-09 07:13:57: Completed 100/1084
2025-12-09 07:14:03: Completed 200/1084
2025-12-09 07:14:10: Completed 300/1084
2025-12-09 07:14:16: Completed 400/1084
2025-12-09 07:14:23: Completed 500/1084
2025-12-09 07:14:30: Completed 600/1084
2025-12-09 07:14:36: Completed 700/1084
2025-12-09 07:14:43: Completed 800/1084
2025-12-09 07:14:50: Completed 900/1084
2025-12-09 07:14:56: Completed 1000/1084
2025-12-09 07:15:02: Saving embeddings
2025-12-09 07:15:02: Embedding Saved
2025-12-09 07:15:02: Ends


# Probability

In [21]:
tokenizer = AutoTokenizer.from_pretrained("yangheng/deberta-v3-base-absa-v1.1")

C:\Users\MissTiny\anaconda3\envs\Creativity\Lib\site-packages\transformers\convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [22]:
model = AutoModelForSequenceClassification.from_pretrained("yangheng/deberta-v3-base-absa-v1.1")

In [23]:
def softmax(logits):
    logits = np.array(logits, dtype=float)
    shifted = logits - np.max(logits)        # stability trick
    exp_vals = np.exp(shifted)
    return exp_vals / np.sum(exp_vals)

In [24]:
def compute_creativity(i,row,tokenizer,model):
    inputs_claude = tokenizer([row.creativity_comment_claude],text_pair=["creativ"], return_tensors="pt")
    with torch.no_grad():
        logits_claude = model(**inputs_claude).logits
    score_claude = softmax(logits_claude[0])[2]


    inputs_gemini = tokenizer([row.creativity_comment_gemini],text_pair=["creativ"], return_tensors="pt")
    with torch.no_grad():
        logits_gemini = model(**inputs_gemini).logits
    score_gemini = softmax(logits_gemini[0])[2]

    inputs_openai = tokenizer([row.creativity_comment],text_pair=["creativ"], return_tensors="pt")
    with torch.no_grad():
        logits_openai = model(**inputs_openai).logits
    score_openai = softmax(logits_openai[0])[2]
    

    row_score = [score_claude,score_gemini,score_openai]
    
    return i, np.mean(row_score)

In [25]:
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,gold_set.shape[0])
N = min(number_size, gold_set.shape[0]-range_start)
max_workers = 20
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
while range_start < gold_set.shape[0]:
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Now at {range_start}")
    creativity = np.zeros(N)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(compute_creativity, i, 
                    gold_set.iloc[i],
                    tokenizer,
                    model): i
            for i in range(range_start,range_end)
        }
        
        completed = 0
        for fut in as_completed(futures):
            i,row_creativity = fut.result()
            creativity[i-range_start]  = row_creativity
            completed += 1
            if completed % 100 == 0:
                print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Completed {completed}/{gold_set.shape[0]}")
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Saving embeddings")
    np.save(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\creativity_65_full_prob.npy", creativity)
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Embedding Saved")
    range_start = range_start + N
    range_end = min(range_start+number_size,gold_set.shape[0])
    N = min(number_size, gold_set.shape[0]-range_start)
    
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

2025-12-16 06:54:04: Start
2025-12-16 06:54:04: Now at 0


C:\Users\MissTiny\AppData\Local\Temp\ipykernel_44512\1495486254.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  logits = np.array(logits, dtype=float)


2025-12-16 06:54:40: Completed 100/2723
2025-12-16 06:55:15: Completed 200/2723
2025-12-16 06:55:47: Completed 300/2723
2025-12-16 06:56:21: Completed 400/2723
2025-12-16 06:56:55: Completed 500/2723
2025-12-16 06:57:29: Completed 600/2723
2025-12-16 06:58:03: Completed 700/2723
2025-12-16 06:58:38: Completed 800/2723
2025-12-16 06:59:13: Completed 900/2723
2025-12-16 06:59:46: Completed 1000/2723
2025-12-16 07:00:23: Completed 1100/2723
2025-12-16 07:00:58: Completed 1200/2723
2025-12-16 07:01:32: Completed 1300/2723
2025-12-16 07:02:09: Completed 1400/2723
2025-12-16 07:02:44: Completed 1500/2723
2025-12-16 07:03:19: Completed 1600/2723
2025-12-16 07:03:52: Completed 1700/2723
2025-12-16 07:04:26: Completed 1800/2723
2025-12-16 07:05:01: Completed 1900/2723
2025-12-16 07:05:37: Completed 2000/2723
2025-12-16 07:06:14: Completed 2100/2723
2025-12-16 07:06:48: Completed 2200/2723
2025-12-16 07:07:26: Completed 2300/2723
2025-12-16 07:08:06: Completed 2400/2723
2025-12-16 07:08:42: Comp

# Test

In [51]:
gold_set.columns

Index(['artwork id', 'title', 'first', 'last', 'workyear from', 'nationality',
       'artistic_value_answer_claude', 'artistic_value_comment_claude',
       'creativity_answer_claude', 'creativity_comment_claude',
       'artistic_value_answer_gemini', 'artistic_value_comment_gemini',
       'creativity_answer_gemini', 'creativity_comment_gemini',
       'artistic_value_answer', 'artistic_value_comment', 'creativity_answer',
       'creativity_comment', 'creative_consist', 'artistic_consist',
       'overall_consist', 'embed_artistic_consist', 'embed_creative_consist',
       'embed_overall_consist'],
      dtype='object')

In [6]:
scores=[]
for i in range(gold_set.shape[0]):
    row = gold_set.iloc[i]
    score_claude= classifier(row.creativity_comment_claude, text_pair="creativ")
    score_gemini=classifier(row.creativity_comment_gemini, text_pair="creativ")
    score_openai=classifier(row.creativity_comment, text_pair="creativ")
    break
    row_score = []
    if score_claude[0]['label']=='Positive':
        row_score.append(score_claude[0]['score'])
    else:
        row_score.append(0)
    if score_gemini[0]['label']=='Positive':
        row_score.append(score_gemini[0]['score'])
    else:
        row_score.append(0)
    if score_openai[0]['label']=='Positive':
        row_score.append(score_openai[0]['score'])
    else:
        row_score.append(0)
    scores.append(np.mean(row_score))
    if i==100:
        break

In [44]:
classifier(row.creativity_comment_claude, text_pair="creativ")

[{'label': 'Positive', 'score': 0.7599267363548279}]

In [60]:
tokenizer = AutoTokenizer.from_pretrained("yangheng/deberta-v3-base-absa-v1.1")
inputs = tokenizer([row.creativity_comment_claude],text_pair=["creativ"], return_tensors="pt")

In [52]:
model = AutoModelForSequenceClassification.from_pretrained("yangheng/deberta-v3-base-absa-v1.1")

In [53]:
with torch.no_grad():
    logits = model(**inputs).logits

In [54]:
logits

tensor([[-1.1991, -0.1684,  1.2890]])

In [55]:
def softmax(logits):
    logits = np.array(logits, dtype=float)
    shifted = logits - np.max(logits)        # stability trick
    exp_vals = np.exp(shifted)
    return exp_vals / np.sum(exp_vals)

In [56]:
softmax(logits[0])

C:\Users\MissTiny\AppData\Local\Temp\ipykernel_39092\1495486254.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  logits = np.array(logits, dtype=float)


array([0.06312346, 0.17694937, 0.75992717])

In [42]:
score_claude= classifier(row.creativity_comment_claude, text_pair="creativ")

In [43]:
score_claude

[{'label': 'Positive', 'score': 0.7599267363548279}]

In [33]:
model.config.id2label

{0: 'Negative', 1: 'Neutral', 2: 'Positive'}

In [34]:
tokenizer = AutoTokenizer.from_pretrained("yangheng/deberta-v3-base-absa-v1.1")

In [35]:
model = AutoModelForSequenceClassification.from_pretrained(
    "yangheng/deberta-v3-base-absa-v1.1")

In [ ]:
creative_sentiment=[]
for i in score:
    if i['label']=='Positive':
        creative_sentiment.append(i['score'])
    elif i['label']=='Negative':
        creative_sentiment.append(-i['score'])
    else:
        creative_sentiment.append(0)